# Parametric PINN simulation

This notebook keeps the full simulation workflow and updates it to train on **only** `(phi1, phi2)`, while `mx,my,k` are fixed.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pinn_ndof_chain_parametric_tf2 import (
    lhs_sample,
    ParametricPIPNNs,
    find_impact_times_parametric,
    propagate_ics_parametric,
)


## 1) Simulation and parameter setup


In [ ]:
# ---- physical/segment settings ----
n_dof = 20
T_seg = 1.0
N_col = 1000
D = 1.0
r = 1.0
c_damp = 0.0
layers = [1 + 5, 64, 64, n_dof]  # internal input: [t,mx,my,k,phi1,phi2]
hyp_ini_weight_loss = [1.0, 1.0]

# ---- reduced parameterization ----
param_names = ['phi1', 'phi2']
fixed_params = {'mx': 1.0, 'my': 0.3, 'k': 1.0}
lb_params = np.array([1.0, 10.0], dtype=np.float32)
ub_params = np.array([2.0, 20.0], dtype=np.float32)
n_cases = 20
params_cases = lhs_sample(n_cases, lb_params, ub_params, seed=42)


## 2) Initial conditions for all training cases


In [ ]:
phi_offsets = np.zeros((n_cases,), dtype=np.float32)
x0_cases = np.zeros((n_cases, n_dof), dtype=np.float32)
xt0_cases = np.zeros((n_cases, n_dof), dtype=np.float32)
y0_cases = np.zeros((n_cases, n_dof), dtype=np.float32)
yt0_cases = np.zeros((n_cases, n_dof), dtype=np.float32)


## 3) Build and train one segment model


In [ ]:
model = ParametricPIPNNs(
    n_dof=n_dof,
    params_cases=params_cases,
    lb_params=lb_params,
    ub_params=ub_params,
    c_damp=c_damp,
    T_seg=T_seg,
    N_col=N_col,
    phi_offsets=phi_offsets,
    x0_cases=x0_cases,
    xt0_cases=xt0_cases,
    y0_cases=y0_cases,
    yt0_cases=yt0_cases,
    D=D,
    layers=layers,
    hyp_ini_weight_loss=hyp_ini_weight_loss,
    optimizer_LB=True,
    fixed_params=fixed_params,
    param_names=param_names,
)

# Optional training
# model.train(nIter=1000, optimizer_LB=True)


## 4) Quick prediction check


In [ ]:
# test on one case index from the sampled set
case_idx = 0
t_eval = np.linspace(0.0, T_seg, 1001)
x_pred, xt_pred, xtt_pred = model.predict(t_eval, case_idx)

print('x_pred:', x_pred.shape, 'xt_pred:', xt_pred.shape, 'xtt_pred:', xtt_pred.shape)


## 5) Impact-time detection for one case


In [ ]:
t_impacts, hit = find_impact_times_parametric(model, case_idx=0, n_scan=500)
print('first impact times:', t_impacts[:5])
print('hit flags:', hit[:5])


## 6) Plot last DOF response


In [ ]:
plt.figure(figsize=(8,4))
plt.plot(t_eval, x_pred[:, -1], lw=1.5)
plt.xlabel('t')
plt.ylabel('x_last')
plt.title('Predicted displacement of last DOF')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
